# recs_017 — D5 options 1, 1b, 2: habit/session on full catalog

**You are here: options 1, 1b, 2** — change **retrieval** (and optionally rerank). **Not option 3** (frozen-pool rerank → [`recs_016`](../ranking/recs_016_ranker_embedding_habit_session_pool.ipynb)).

## D5 map (same vectors, different pipeline slot)

| Option | Notebook | What changes | Retrieval | Ranking | Status |
|--------|----------|--------------|-----------|---------|--------|
| **Shipped** | eval job | baseline | `two_tower_v1` → top-100 | D1 | **live** |
| **3** | [`recs_016`](../ranking/recs_016_ranker_embedding_habit_session_pool.ipynb) | rank only | `two_tower_v1` → top-100 *(same)* | USE on pool | **killed** |
| **1** | **this notebook §5–6** | new retriever + rerank | habit → catalog top-M | session reranks M | **killed** (~0.034 NDCG@10) |
| **1b** | **this notebook §6** | new single-stage | — | fused query → catalog top-K | **killed** (~0.037 NDCG@10) |
| **2** | this notebook + export | new retriever, old ranker | habit → export top-100 | D1 on new pools | **killed** (not run; §5 below TT @100) |
| **4–5** | — | retrain tower | deferred | deferred | — |

### What each option in *this* notebook tests

| Option | Pattern | Primary metric | Compare to |
|--------|---------|----------------|------------|
| **1** | Cascade: `score(u_habit, catalog)` → top-M → rerank by `q_session` | Recall/Hit **@M=100** (§5), then NDCG **@10** (§6) | `two_tower_v1` **@100** in `eval_retrieval_overall.csv` |
| **1b** | Single fused query over full catalog | NDCG **@10** (§6) | `fusion_c`, `raw` anchors |
| **2** | Export habit top-100 → run D1 | End-to-end NDCG **@10** vs shipped stack | D1 on `two_tower_v1` pools — **not wired yet** |

### Why this notebook reports *both* retrieval and ranking

Same split as `recs_job_eval_retrieval.py` (`eval_retrieval_*.csv` vs `eval_ranking_*.csv`):

| Section | Depth | Metrics | Question |
|---------|-------|---------|----------|
| **§5 Stage-1** | **M=100** (`M_STAGE1`) | Recall@M, Hit@M | Can habit vectors **retrieve** positives into a pool? Compare to `two_tower_v1` **@100** in `eval_retrieval_overall.csv`. |
| **§6 Stage-2 + anchors** | **K=10** (`K_FINAL`) | NDCG@K, Hit@K, MAP, MRR, … | After cascade or single-stage sort, what is **final list** quality? |

**Do not read bare `two_tower_v1` Hit@10 here as “retrieval failed.”** Two-tower is strong at **pool recall @100** (~0.51 Hit) and weak at **top-10 order without D1** (~0.05 Hit) — see `recs_013` and `eval_retrieval_overall.csv` vs `eval_ranking_overall.csv`.

Anchors in §6 (`raw`, `popularity_train`, `fusion_c`, `two_tower_v1`) are full-catalog @10 baselines for context, not the shipped stack (`two_tower_v1` pool + D1 rerank).

### Folder / name note

This file lives under `notebooks/retrieval/` because the **primary bet** is new **retrieval** vectors (option 2 export). It also runs **rerank** steps (options 1 / 1b), so a clearer name might be `recs_017_habit_session_full_catalog_cascade.ipynb` under `notebooks/ranking/` — **not moved yet**; links still point here.

**Status:** **killed** — options 1/1b lose to `fusion_c` @10; stage-1 @100 ~0.506 vs `two_tower_v1` ~0.512; option 2 not pursued. See [`ranking_decision_log.md`](../../docs/ranking_decision_log.md) § 2026-06-13.

### Habit vs behavior (read this first)

| Vector | What it encodes | How built in this repo |
|--------|-----------------|------------------------|
| **`q_session`** | Current review | `embed(query_text)` |
| **`u_reviews`** | What user *wrote* about past games | Mean embed of train review texts |
| **`u_behavior_playtime`** | Play history (fusion_c) | Playtime-weighted mean of train-app profile vectors (`retrieve.py`) |
| **`u_behavior_equal`** | Play history (ablation) | Equal mean of train-app profile vectors |
| **`habit_fused_*`** | Long-term taste | `normalize(0.5·u_behavior + 0.5·u_reviews)` — **two variants** (playtime vs equal behavior) |

**Both behavior recipes are reported as separate methods** in stage-1 and cascade tables.

### Open questions (decide before full run)

1. ~~**`u_behavior` recipe**~~ → **both** (playtime + equal-mean).
2. **`habit_fused` weights:** fixed 50/50 or tunable on `train_tune`?
3. **Option 2 export vector:** `u_behavior`, `u_reviews`, or `habit_fused` for top-100 pool job?
4. **Anchors:** require beat `two_tower_v1` @100, `fusion_c`, or D1 end-to-end (option 2 only)?
5. **Cascade M:** `M_STAGE1=100` (match `k_retrieval`) or larger stage-1 for recall?

---

Method families:
- Family 1 (incumbent single-stage): `raw`, `popularity_train`, `multi_mean_train`
- Family 2 (single-stage fused): `fused_single_stage = score(normalize(alpha*u_behavior + beta*u_reviews + gamma*q_session), item)`
- Family 3 (retrieval→ranking cascade): `habit_session_two_stage = rerank_topM(score(u_habit, item), q_session)`

Similarity/score note:
- The model computes a similarity value between query/user vector and item vector; that similarity is the item score.
- Items are sorted in descending score order for ranking metrics.

Vector definitions:
- `q_session`: embedding of the current query review text.
- `u_reviews`: pooled embedding of support/train review texts for the user.
- `u_behavior`: pooled embedding of support/train app IDs via the item embedding matrix.
- `u_habit`: retrieval-stage habit vector used for candidate generation (in this notebook: `u_habit = habit_fused = normalize(0.5*u_behavior + 0.5*u_reviews)`).

Pipeline definitions:
- Retrieval stage: candidate generation using habit vector (`u_habit`) against item embeddings; evaluate candidate hit/recall at `M_STAGE1`.
- Ranking stage: rerank retrieval-stage candidates with session/query signal (`q_session` or fused variant); evaluate final ranking metrics at `K_FINAL`.

Decile diagnostics (for **§6 ranking @10** slices only):
- Popularity deciles from mean target train popularity; labels **`D1`–`D10`** (**`D1` = long-tail**, **`D10` = head** — not ranker D1).
- Report per-decile metrics and deltas vs anchor methods to expose head-vs-tail behavior.

Execution order:
1. Setup and config
2. Shared helpers
3. Build eval examples and metadata
4. Build vectors (`u_behavior`, `u_reviews`, `q_session`, fused)
5. Retrieval-stage evaluation
6. Ranking-stage evaluation + incumbent comparison
7. Support/pop-decile slices + baseline deltas
8. Personalization metrics integrated in overall/slice/support/pop-decile tables
9. Write summary artifact

This notebook is intentionally lean and artifact-driven.
Reference docs:
- `docs/recommendation_evaluation_overview.md`
- `docs/archive/recommender_transition_plan.md`

## 1) Setup and Top-Level Config

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

import steam_review_ml.evaluation.retrieval_offline_eval as ev


def _find_repo_root(start: Path) -> Path:
    here = start.resolve()
    for d in [here, *here.parents]:
        if (d / "pyproject.toml").is_file():
            return d
    raise RuntimeError(f"Could not find repo root from start={start}")


# ===== 1) Setup and top-level config =====
REPO_ROOT = _find_repo_root(Path.cwd())
ARTIFACT_DIR = REPO_ROOT / "artifacts" / "recs"
EVAL_DIR = ARTIFACT_DIR / "eval"

SPLIT = "val"
K_FINAL = 10
M_STAGE1 = 100
K_PERSONALIZATION = 10
MAX_EXAMPLES = 12_500
RANDOM_SEED = 2026
MIN_REVIEW_CHARS = 30

# Frozen cohort — same parquet as ranking eval (configs/recs_job_eval_ranking.json).
USE_EXAMPLES_CACHE = True
EVAL_EXAMPLES_PARQUET = ARTIFACT_DIR / "eval_cache" / "val_dev_12k_v1" / "eval_examples.parquet"

# Fallback only when USE_EXAMPLES_CACHE=False or parquet missing (configs/recs_job_eval_retrieval.json).
COHORT_SIZING = {
    ("val_multi_pos_eval", "val_multi_pos_train"): 0.5,
    ("val_multi_pos_eval", "val_pos_train"): 0.25,
    ("val_multi_pos_eval", "val_train"): 0.15,
    ("val_multi_pos_eval", "val_no_train"): 0.1,
}

ALPHA_BEHAVIOR = 0.45
BETA_REVIEWS = 0.45
GAMMA_SESSION = 0.10

# Anchors for ranking comparison (skip multi_mean_train here — re-embeds all support texts, very slow).
METHODS_ANCHOR = ["raw", "popularity_train", "fusion_c_raw_plus_behavior", "two_tower_v1"]
TWO_TOWER_MODEL_PATH = REPO_ROOT / "artifacts/recs/towers/val_dev_12k_v1/updated_user__updated_profile200_item.keras"
SUPPORT_BUCKETS = ["0", "1", "2-3", "4-7", "8+"]
SLICE_RULES = {
    "slice_a_multi_target": "n_eval_targets >= 2",
    "slice_b_single_target": "n_eval_targets == 1",
    "slice_c_zero_target": "n_eval_targets == 0",
}

ENABLE_PLOTS = True

/home/ryanr/miniconda3/envs/tf_condaforge/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2) Shared Helpers

In [9]:
# ===== 2) Shared helpers (single definition source) =====

def l2_normalize(v: np.ndarray) -> np.ndarray:
    arr = np.asarray(v, dtype=np.float32).ravel()
    nrm = float(np.linalg.norm(arr))
    if nrm <= 1e-12:
        return arr
    return (arr / nrm).astype(np.float32)


def support_bucket(n: int) -> str:
    n = int(n)
    if n <= 0:
        return "0"
    if n == 1:
        return "1"
    if n <= 3:
        return "2-3"
    if n <= 7:
        return "4-7"
    return "8+"


def jaccard(a: set[int], b: set[int]) -> float:
    u = a | b
    if not u:
        return 1.0
    return len(a & b) / len(u)


def rank_rows(scores: np.ndarray) -> np.ndarray:
    return np.argsort(-scores)


def scores_from_query(q: np.ndarray, X: np.ndarray, app_to_row: dict[int, int], query_app_id: int) -> np.ndarray:
    s = (X @ q).astype(np.float32)
    row = app_to_row.get(int(query_app_id))
    if row is not None:
        s[row] = -np.inf
    return s


def eval_row(ranked: np.ndarray, positives: set[int], app_ids: np.ndarray, k: int) -> dict[str, float]:
    return {
        "Hit@K": ev.hit_rate_at_k(ranked, positives, k, app_ids),
        "Recall@K": ev.recall_at_k(ranked, positives, k, app_ids),
        "MAP@K": ev.average_precision_at_k(ranked, positives, k, app_ids),
        "NDCG@K": ev.ndcg_at_k(ranked, positives, k, app_ids),
        "MRR": ev.mrr(ranked, positives, app_ids),
    }


def slice_name_from_n_targets(n_eval_targets: int) -> str:
    if n_eval_targets >= 2:
        return "slice_a_multi_target"
    if n_eval_targets == 1:
        return "slice_b_single_target"
    return "slice_c_zero_target"


def example_positives(ex: dict) -> set[int]:
    """Canonical eval positives key (cache + prepare_eval_inputs)."""
    raw = ex.get("validation_positive_app_ids", ex.get("positive_app_ids"))
    if raw is None:
        raise KeyError("example missing validation_positive_app_ids")
    return {int(a) for a in raw}

## 3) Build Evaluation Examples and Metadata

In [3]:
# ===== 3) Build evaluation examples and metadata =====
if USE_EXAMPLES_CACHE and EVAL_EXAMPLES_PARQUET.is_file():
    print(f"Loading eval examples from cache: {EVAL_EXAMPLES_PARQUET}")
    inputs = ev.prepare_eval_inputs_from_cache(
        repo_root=REPO_ROOT,
        split=SPLIT,
        min_review_chars=MIN_REVIEW_CHARS,
        examples_parquet=EVAL_EXAMPLES_PARQUET,
        artifact_dir=ARTIFACT_DIR,
        verbose=True,
    )
    examples_source = "parquet_cache"
else:
    if USE_EXAMPLES_CACHE:
        print(f"Cache not found at {EVAL_EXAMPLES_PARQUET}; falling back to prepare_eval_inputs().")
    else:
        print("USE_EXAMPLES_CACHE=False; using prepare_eval_inputs().")
    inputs = ev.prepare_eval_inputs(
        repo_root=REPO_ROOT,
        split=SPLIT,
        active_cohort="all",
        max_examples=MAX_EXAMPLES,
        support_app_filter_mode="strict",
        cohort_sizing=COHORT_SIZING,
        min_review_chars=MIN_REVIEW_CHARS,
        max_train_rows_per_user=5,
        random_seed=RANDOM_SEED,
        artifact_dir=ARTIFACT_DIR,
        verbose=True,
    )
    examples_source = "prepare_eval_inputs"

examples = inputs.examples
X = inputs.embedding_matrix
app_ids = inputs.app_ids
app_to_row = inputs.app_to_row
retriever = inputs.retriever

example_meta = pd.DataFrame(
    {
        "ex_idx": np.arange(len(examples), dtype=int),
        "user_id": [str(ex["user_id"]) for ex in examples],
        "query_app_id": [int(ex["query_app_id"]) for ex in examples],
        "n_eval_targets": [int(ex["n_eval_targets"]) for ex in examples],
        "n_support_train": [int(len(ex.get("train_review_rows", []))) for ex in examples],
    }
)
example_meta["slice_name"] = example_meta["n_eval_targets"].map(slice_name_from_n_targets)
example_meta["train_support_bucket"] = example_meta["n_support_train"].map(support_bucket)

print(f"examples: {len(examples)} source={examples_source}")
display(example_meta.head())

Loading eval examples from cache: /home/ryanr/workspace/steam_recommendations/artifacts/recs/eval_cache/val_dev_12k_v1/eval_examples.parquet
Loaded 12,500 cached eval examples from /home/ryanr/workspace/steam_recommendations/artifacts/recs/eval_cache/val_dev_12k_v1/eval_examples.parquet
examples: 12500 source=parquet_cache


,ex_idx,user_id,query_app_id,n_eval_targets,n_support_train,slice_name,train_support_bucket
0,0,76561198001296435,812140,1,0,slice_b_single_target,0
1,1,76561198006360052,485510,1,0,slice_b_single_target,0
2,2,76561198094909231,646570,2,0,slice_a_multi_target,0
3,3,76561198133278614,4000,1,0,slice_b_single_target,0
4,4,76561198073849336,582010,1,0,slice_b_single_target,0


In [4]:
pd.DataFrame(examples).head()

,user_id,query_app_id,query_text,query_ts,validation_positive_app_ids,n_eval_targets,train_review_rows,cohort,eval_pos_cohort
0,76561198001296435,812140,"> Got it on Sale\n> Started it up, played Kass...",1.568428e+09,{262060},1,"[{'app_id': 359550, 'text': 'Facepalm as your ...",val_multi_pos_train,val_multi_pos_eval
1,76561198006360052,485510,Pretty fun so far. Took me about 5 hours to b...,1.602431e+09,{552500},1,"[{'app_id': 899440, 'text': 'As a long term Mo...",val_multi_pos_train,val_multi_pos_eval
2,76561198094909231,646570,A game about setting up your broken-a.-f. card...,1.602966e+09,"{40800, 435150}",2,"[{'app_id': 8870, 'text': 'A modern classic fo...",val_multi_pos_train,val_multi_pos_eval
3,76561198133278614,4000,You can do almost anything in this game 10/10,1.528897e+09,{413150},1,"[{'app_id': 253230, 'text': 'Man i cna't tell ...",val_multi_pos_train,val_multi_pos_eval
4,76561198073849336,582010,My first Monster Hunter was MH3: Tri. I was wa...,1.535904e+09,{814380},1,"[{'app_id': 221640, 'text': '*Music starts* U...",val_multi_pos_train,val_multi_pos_eval


In [5]:
example_meta['slice_name'].value_counts()

slice_name
slice_b_single_target    11775
slice_a_multi_target       725
Name: count, dtype: int64

## 4) Build Vectors (both `u_behavior` recipes + habit fused variants)

In [6]:
# ===== 4) Build vectors — both u_behavior recipes =====
from steam_review_ml.recommender.retrieve import (
    _mean_train_review_text_embedding,
    _weighted_mean_behavior_embedding,
    fusion_c_raw_plus_behavior_query_vector,
)

vector_rows = []
for ex_idx, ex in enumerate(examples):
    q_session = np.asarray(retriever.embed_text(str(ex["query_text"])), dtype=np.float32)
    u_reviews = _mean_train_review_text_embedding(retriever, ex)

    u_behavior_playtime, _ = _weighted_mean_behavior_embedding(
        ex,
        embedding_matrix=X,
        app_to_row=app_to_row,
        fallback=u_reviews,
    )

    support_app_ids = sorted(
        {int(r["app_id"]) for r in ex.get("train_review_rows", []) if int(r["app_id"]) in app_to_row}
    )
    if support_app_ids:
        emb = np.stack([X[app_to_row[a]] for a in support_app_ids], axis=0).astype(np.float32)
        u_behavior_equal = l2_normalize(emb.mean(axis=0))
    else:
        u_behavior_equal = u_reviews

    habit_fused_playtime = l2_normalize(0.5 * u_behavior_playtime + 0.5 * u_reviews)
    habit_fused_equal = l2_normalize(0.5 * u_behavior_equal + 0.5 * u_reviews)

    q_fusion_c = fusion_c_raw_plus_behavior_query_vector(
        retriever, ex, embedding_matrix=X, app_to_row=app_to_row
    )
    q_fused_blend_playtime = l2_normalize(
        ALPHA_BEHAVIOR * u_behavior_playtime + BETA_REVIEWS * u_reviews + GAMMA_SESSION * q_session
    )
    q_fused_blend_equal = l2_normalize(
        ALPHA_BEHAVIOR * u_behavior_equal + BETA_REVIEWS * u_reviews + GAMMA_SESSION * q_session
    )

    vector_rows.append(
        {
            "ex_idx": ex_idx,
            "q_session": q_session,
            "u_reviews": u_reviews,
            "u_behavior_playtime": u_behavior_playtime,
            "u_behavior_equal": u_behavior_equal,
            "habit_fused_playtime": habit_fused_playtime,
            "habit_fused_equal": habit_fused_equal,
            "q_fusion_c": q_fusion_c,
            "q_fused_blend_playtime": q_fused_blend_playtime,
            "q_fused_blend_equal": q_fused_blend_equal,
        }
    )
    if (ex_idx + 1) % 2000 == 0:
        print(f"  vectorized {ex_idx + 1:,}/{len(examples):,}", flush=True)

vector_store = {int(r["ex_idx"]): r for r in vector_rows}
print("vectorized examples:", len(vector_store))

2026-06-12 06:44:15.371491: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1781261055.387047   35127 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1781261055.392586   35127 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1781261055.447437   35127 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781261055.447465   35127 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781261055.447468   35127 computation_placer.cc:177] computation placer alr

  vectorized 2,000/12,500
  vectorized 4,000/12,500
  vectorized 6,000/12,500
  vectorized 8,000/12,500
  vectorized 10,000/12,500
  vectorized 12,000/12,500
vectorized examples: 12500


## 5) Retrieval-Stage Evaluation (Candidate Generation at M)

In [10]:
# ===== 5) Retrieval-stage evaluation (candidate generation at M) =====
stage1_method_vectors = {
    "stage1_u_behavior_playtime": "u_behavior_playtime",
    "stage1_u_behavior_equal": "u_behavior_equal",
    "stage1_u_reviews": "u_reviews",
    "stage1_habit_fused_playtime": "habit_fused_playtime",
    "stage1_habit_fused_equal": "habit_fused_equal",
}

stage1_rows = []
for ex_idx, ex in enumerate(examples):
    positives = example_positives(ex)
    if not positives:
        continue
    for method_name, vec_key in stage1_method_vectors.items():
        q = vector_store[ex_idx][vec_key]
        ranked = rank_rows(scores_from_query(q, X, app_to_row, int(ex["query_app_id"])))
        topM = ranked[:M_STAGE1]
        rec_m = ev.recall_at_k(topM, positives, M_STAGE1, app_ids)
        hit_m = ev.hit_rate_at_k(topM, positives, M_STAGE1, app_ids)
        stage1_rows.append(
            {
                "method": method_name,
                "ex_idx": ex_idx,
                "Recall@M": rec_m,
                "Hit@M": hit_m,
            }
        )

stage1_per_example = pd.DataFrame(stage1_rows)
stage1_table = (
    stage1_per_example.groupby("method", observed=True)[["Recall@M", "Hit@M"]]
    .mean()
    .reset_index()
    .sort_values(["Recall@M", "Hit@M"], ascending=False)
)

display(stage1_table)

,method,Recall@M,Hit@M
3,stage1_u_behavior_playtime,0.487246,0.50560
2,stage1_u_behavior_equal,0.484793,0.50280
0,stage1_habit_fused_equal,0.484683,0.50256
1,stage1_habit_fused_playtime,0.484062,0.50192
4,stage1_u_reviews,0.452953,0.47120


## 6) Ranking-stage @K (`eval_ranking_*` analogue)

Cascade rerank (option 1), single-stage fusion (option 1b), and full-catalog anchors @10. **`two_tower_v1` here is bare top-10 order**, not retrieval @100 and not pool + D1.

In [11]:
# ===== 6) Ranking-stage evaluation (rerank) + anchor baselines =====
rng = np.random.default_rng(RANDOM_SEED)
anchor_registry = ev._build_method_registry(
    retriever=retriever,
    X=X,
    pop_row=inputs.pop_row,
    app_to_row=app_to_row,
    multi_max_reviews=5,
    rng=rng,
    mask_query_app=True,
    two_tower_model_path=TWO_TOWER_MODEL_PATH,
)
anchor_registry = {m: anchor_registry[m] for m in METHODS_ANCHOR}

SINGLE_STAGE_CUSTOM = {
    "single_fused_blend_equal": "q_fused_blend_equal",
    "single_fused_blend_playtime": "q_fused_blend_playtime",
}

TWO_STAGE_SOURCES = {
    "two_stage_behavior_equal_session": "u_behavior_equal",
    "two_stage_behavior_playtime_session": "u_behavior_playtime",
    "two_stage_reviews_session": "u_reviews",
    "two_stage_habit_fused_equal_session": "habit_fused_equal",
    "two_stage_habit_fused_playtime_session": "habit_fused_playtime",
}


def _single_stage_scores(ex_idx: int, ex: dict) -> dict[str, np.ndarray]:
    """Anchors + notebook single-stage fused blends (both behavior recipes)."""
    vs = vector_store[ex_idx]
    qid = int(ex["query_app_id"])
    out = {m: anchor_registry[m](ex) for m in METHODS_ANCHOR}
    for method_name, vec_key in SINGLE_STAGE_CUSTOM.items():
        out[method_name] = scores_from_query(vs[vec_key], X, app_to_row, qid)
    return out


def _two_stage_rerank_order(ex_idx: int, ex: dict, q_stage1: np.ndarray) -> np.ndarray:
    """Retrieve top-M from stage-1 vector and rerank those candidates by session."""
    s1 = scores_from_query(q_stage1, X, app_to_row, int(ex["query_app_id"]))
    cands = rank_rows(s1)[:M_STAGE1]
    q_session = vector_store[ex_idx]["q_session"]
    s2 = scores_from_query(q_session, X, app_to_row, int(ex["query_app_id"]))
    return cands[np.argsort(-s2[cands])]


def _score_fn_for_method(method_name: str):
    """Build a scorer callable that mirrors each method's inference behavior."""
    if method_name in METHODS_ANCHOR:
        return lambda ex, m=method_name: anchor_registry[m](ex)

    if method_name in SINGLE_STAGE_CUSTOM:
        vec_key = SINGLE_STAGE_CUSTOM[method_name]

        def _single(ex, _k=vec_key):
            vs = vector_store[int(ex["_ex_idx"])]
            return scores_from_query(vs[_k], X, app_to_row, int(ex["query_app_id"]))

        return _single

    if method_name in TWO_STAGE_SOURCES:
        vec_key = TWO_STAGE_SOURCES[method_name]

        def _cascade(ex, _k=vec_key):
            ex_idx = int(ex["_ex_idx"])
            qid = int(ex["query_app_id"])
            q_stage1 = vector_store[ex_idx][_k]
            q_session = vector_store[ex_idx]["q_session"]
            s1 = scores_from_query(q_stage1, X, app_to_row, qid)
            cands = rank_rows(s1)[:M_STAGE1]
            s2 = scores_from_query(q_session, X, app_to_row, qid)
            out = np.full_like(s2, -np.inf)
            out[cands] = s2[cands]
            return out

        return _cascade

    raise KeyError(method_name)


def _examples_for_personalization(examples: list[dict]) -> list[dict]:
    """Attach example indices so scorer callables can access vector_store."""
    out = []
    for ex_idx, ex in enumerate(examples):
        ex_copy = dict(ex)
        ex_copy["_ex_idx"] = ex_idx
        out.append(ex_copy)
    return out


rows = []
for ex_idx, ex in enumerate(examples):
    positives = example_positives(ex)
    if not positives:
        continue

    for method_name, s in _single_stage_scores(ex_idx, ex).items():
        ranked = rank_rows(s)
        metric_vals = eval_row(ranked, positives, app_ids, K_FINAL)
        if method_name in METHODS_ANCHOR:
            family = "anchor"
        else:
            family = "single_fused"
        rows.append(
            {
                "method": method_name,
                "family": family,
                "ex_idx": ex_idx,
                **metric_vals,
            }
        )

    for method_name, q_stage1 in (
        (m, vector_store[ex_idx][k]) for m, k in TWO_STAGE_SOURCES.items()
    ):
        rerank_order = _two_stage_rerank_order(ex_idx, ex, q_stage1)
        metric_vals = eval_row(rerank_order, positives, app_ids, K_FINAL)
        rows.append(
            {
                "method": method_name,
                "family": "two_stage",
                "ex_idx": ex_idx,
                **metric_vals,
            }
        )

per_example_table = pd.DataFrame(rows).merge(example_meta, on="ex_idx", how="left")

overall_table = (
    per_example_table.groupby(["family", "method"], observed=True)[["Hit@K", "Recall@K", "MAP@K", "NDCG@K", "MRR"]]
    .mean()
    .reset_index()
    .sort_values(["NDCG@K", "MAP@K", "MRR"], ascending=False)
)

by_slice_table = (
    per_example_table.groupby(["slice_name", "family", "method"], observed=True)[["Hit@K", "Recall@K", "MAP@K", "NDCG@K", "MRR"]]
    .mean()
    .reset_index()
)

all_methods = sorted(per_example_table["method"].unique().tolist())
examples_for_personalization = _examples_for_personalization(examples)
methods_for_personalization = {m: _score_fn_for_method(m) for m in all_methods}

personalization_table = ev._table_personalization(
    methods=methods_for_personalization,
    examples=examples_for_personalization,
    X=X,
    app_ids=app_ids,
    pop_row=inputs.pop_row,
    k_personalization=K_PERSONALIZATION,
    verbose=False,
)

slice_personalization_rows = []
for slice_name, g in per_example_table.groupby("slice_name", observed=True):
    ex_indices = sorted({int(x) for x in g["ex_idx"].tolist()})
    p = ev._table_personalization(
        methods=methods_for_personalization,
        examples=examples_for_personalization,
        X=X,
        app_ids=app_ids,
        pop_row=inputs.pop_row,
        k_personalization=K_PERSONALIZATION,
        example_indices=ex_indices,
        verbose=False,
    ).copy()
    p["slice_name"] = str(slice_name)
    slice_personalization_rows.append(p)

slice_personalization = pd.concat(slice_personalization_rows, ignore_index=True, sort=False)

overall_table = overall_table.merge(personalization_table, on="method", how="left")
by_slice_table = by_slice_table.merge(slice_personalization, on=["method", "slice_name"], how="left")

display(overall_table.head(20))

2026-06-12 08:50:00.363321: E tensorflow/core/util/util.cc:131] oneDNN supports DT_INT64 only on platforms with AVX-512. Falling back to the default Eigen-based implementation if present.


,family,method,Hit@K,Recall@K,MAP@K,NDCG@K,MRR,ILD@10,CatalogCoverage@10,Novelty@10,PersonalizationGapVsPopularity@10
0,anchor,popularity_train,0.15112,0.146796,0.050594,0.073109,0.073756,0.212335,0.034921,5.317471,0.000000
1,anchor,fusion_c_raw_plus_behavior,0.08536,0.079430,0.025877,0.038836,0.042861,0.134907,1.000000,9.653051,0.974458
2,single_fused,single_fused_blend_playtime,0.08504,0.078417,0.023790,0.037044,0.040830,0.132215,1.000000,9.611407,0.972101
3,single_fused,single_fused_blend_equal,0.08440,0.077917,0.023608,0.036754,0.040572,0.132160,1.000000,9.626320,0.973040
4,anchor,raw,0.07168,0.066606,0.024917,0.035020,0.040381,0.160723,1.000000,9.838045,0.979470
5,two_stage,two_stage_behavior_equal_session,0.07232,0.067265,0.023890,0.034341,0.037360,0.139535,1.000000,9.840590,0.980040
6,two_stage,two_stage_habit_fused_equal_session,0.07152,0.066284,0.023849,0.034118,0.037246,0.142911,1.000000,9.842596,0.979350
7,two_stage,two_stage_behavior_playtime_session,0.07160,0.066611,0.023767,0.034095,0.037221,0.139563,1.000000,9.838497,0.979956
8,two_stage,two_stage_habit_fused_playtime_session,0.07120,0.065984,0.023589,0.033857,0.036975,0.142909,1.000000,9.842393,0.979321
9,two_stage,two_stage_reviews_session,0.07072,0.065008,0.023098,0.033287,0.035947,0.148891,1.000000,9.841024,0.979007


## 7) Cross-Sections: Support Buckets, Pop Deciles, and Deltas

In [12]:
# ===== 7) Cross-sections: support buckets, pop deciles, and deltas =====

def _build_ex_pop_table(examples: list[dict], app_ids: np.ndarray, pop_row: np.ndarray) -> pd.DataFrame:
    """Compute mean target popularity per example and assign popularity deciles."""
    app_pop = {int(a): float(c) for a, c in zip(app_ids, pop_row)}
    rows = []
    for ex_idx, ex in enumerate(examples):
        vals = [app_pop.get(int(a), 0.0) for a in example_positives(ex)]
        rows.append({"ex_idx": ex_idx, "pos_pop_mean": float(np.mean(vals)) if vals else np.nan})
    ex_pop = pd.DataFrame(rows)
    valid = ex_pop["pos_pop_mean"].notna()
    if valid.sum() > 0:
        ex_pop.loc[valid, "pos_pop_decile"] = pd.qcut(
            ex_pop.loc[valid, "pos_pop_mean"], q=10, labels=[f"D{i}" for i in range(1, 11)], duplicates="drop"
        )
    return ex_pop


def _personalization_by_group(
    per_example_table: pd.DataFrame,
    group_col: str,
    methods_for_personalization: dict,
    examples_for_personalization: list[dict],
) -> pd.DataFrame:
    """Compute personalization table at the same grouping level as ranking metrics."""
    rows = []
    for group_val, g in per_example_table.groupby(group_col, observed=True):
        ex_indices = sorted({int(x) for x in g["ex_idx"].tolist()})
        p = ev._table_personalization(
            methods=methods_for_personalization,
            examples=examples_for_personalization,
            X=X,
            app_ids=app_ids,
            pop_row=inputs.pop_row,
            k_personalization=K_PERSONALIZATION,
            example_indices=ex_indices,
            verbose=False,
        ).copy()
        p[group_col] = group_val
        rows.append(p)
    return pd.concat(rows, ignore_index=True, sort=False)


def _build_delta_vs_baselines(overall_table: pd.DataFrame) -> pd.DataFrame:
    """Compute method deltas vs raw and popularity anchors."""
    anchors = overall_table[overall_table["method"].isin(["raw", "popularity_train"])][["method", "Hit@K", "NDCG@K", "MRR"]]
    anchor_map = {r["method"]: r for _, r in anchors.iterrows()}
    rows = []
    for _, r in overall_table.iterrows():
        for anchor in ["raw", "popularity_train"]:
            if anchor not in anchor_map:
                continue
            a = anchor_map[anchor]
            rows.append(
                {
                    "method": r["method"],
                    "family": r["family"],
                    "anchor": anchor,
                    "Hit@10_delta_vs_anchor": float(r["Hit@K"] - a["Hit@K"]),
                    "NDCG@10_delta_vs_anchor": float(r["NDCG@K"] - a["NDCG@K"]),
                    "MRR_delta_vs_anchor": float(r["MRR"] - a["MRR"]),
                }
            )
    return pd.DataFrame(rows)


by_support_table = (
    per_example_table.groupby(["train_support_bucket", "family", "method"], observed=True)[["Hit@K", "Recall@K", "MAP@K", "NDCG@K", "MRR"]]
    .mean()
    .reset_index()
)

ex_pop = _build_ex_pop_table(examples, app_ids, inputs.pop_row)

by_pop_decile_table = (
    per_example_table.merge(ex_pop[["ex_idx", "pos_pop_decile"]], on="ex_idx", how="left")
    .dropna(subset=["pos_pop_decile"])
    .groupby(["pos_pop_decile", "family", "method"], observed=True)[["Hit@K", "Recall@K", "MAP@K", "NDCG@K", "MRR"]]
    .mean()
    .reset_index()
)

support_personalization = _personalization_by_group(
    per_example_table=per_example_table,
    group_col="train_support_bucket",
    methods_for_personalization=methods_for_personalization,
    examples_for_personalization=examples_for_personalization,
)

per_ex_pop = per_example_table.merge(ex_pop[["ex_idx", "pos_pop_decile"]], on="ex_idx", how="left")
pop_personalization = _personalization_by_group(
    per_example_table=per_ex_pop.dropna(subset=["pos_pop_decile"]),
    group_col="pos_pop_decile",
    methods_for_personalization=methods_for_personalization,
    examples_for_personalization=examples_for_personalization,
)

by_support_table = by_support_table.merge(support_personalization, on=["method", "train_support_bucket"], how="left")
by_pop_decile_table = by_pop_decile_table.merge(pop_personalization, on=["method", "pos_pop_decile"], how="left")

delta_vs_baselines_table = _build_delta_vs_baselines(overall_table)

display(by_support_table.head(20))
display(by_pop_decile_table.head(20))
display(delta_vs_baselines_table.head(20))

,train_support_bucket,family,method,Hit@K,Recall@K,MAP@K,NDCG@K,MRR,ILD@10,CatalogCoverage@10,Novelty@10,PersonalizationGapVsPopularity@10
0,0,anchor,fusion_c_raw_plus_behavior,0.08536,0.079430,0.025877,0.038836,0.042861,0.134907,1.000000,9.653051,0.974458
1,0,anchor,popularity_train,0.15112,0.146796,0.050594,0.073109,0.073756,0.212335,0.034921,5.317471,0.000000
2,0,anchor,raw,0.07168,0.066606,0.024917,0.035020,0.040381,0.160723,1.000000,9.838045,0.979470
3,0,anchor,two_tower_v1,0.04680,0.043740,0.010325,0.018161,0.026419,0.261639,0.987302,12.167556,0.995623
4,0,single_fused,single_fused_blend_equal,0.08440,0.077917,0.023608,0.036754,0.040572,0.132160,1.000000,9.626320,0.973040
5,0,single_fused,single_fused_blend_playtime,0.08504,0.078417,0.023790,0.037044,0.040830,0.132215,1.000000,9.611407,0.972101
6,0,two_stage,two_stage_behavior_equal_session,0.07232,0.067265,0.023890,0.034341,0.037360,0.139535,1.000000,9.840590,0.980040
7,0,two_stage,two_stage_behavior_playtime_session,0.07160,0.066611,0.023767,0.034095,0.037221,0.139563,1.000000,9.838497,0.979956
8,0,two_stage,two_stage_habit_fused_equal_session,0.07152,0.066284,0.023849,0.034118,0.037246,0.142911,1.000000,9.842596,0.979350
9,0,two_stage,two_stage_habit_fused_playtime_session,0.07120,0.065984,0.023589,0.033857,0.036975,0.142909,1.000000,9.842393,0.979321


,pos_pop_decile,family,method,Hit@K,Recall@K,MAP@K,NDCG@K,MRR,ILD@10,CatalogCoverage@10,Novelty@10,PersonalizationGapVsPopularity@10
0,D1,anchor,fusion_c_raw_plus_behavior,0.122709,0.118659,0.040639,0.059058,0.057381,0.133156,0.993651,9.733712,0.980234
1,D1,anchor,popularity_train,0.000000,0.000000,0.000000,0.000000,0.005287,0.212278,0.034921,5.314266,0.000000
2,D1,anchor,raw,0.102789,0.101195,0.037011,0.052016,0.052139,0.159045,1.000000,9.898705,0.982913
3,D1,anchor,two_tower_v1,0.101992,0.099469,0.027076,0.043887,0.043248,0.256939,0.942857,12.170429,0.996755
4,D1,single_fused,single_fused_blend_equal,0.125100,0.121580,0.036508,0.056506,0.053737,0.130497,0.993651,9.697603,0.979404
5,D1,single_fused,single_fused_blend_playtime,0.123506,0.119456,0.036888,0.056434,0.054450,0.130661,0.993651,9.678158,0.978517
6,D1,two_stage,two_stage_behavior_equal_session,0.105179,0.101992,0.038761,0.053702,0.052373,0.138434,0.993651,9.929790,0.984156
7,D1,two_stage,two_stage_behavior_playtime_session,0.102789,0.099602,0.037985,0.052561,0.051646,0.138216,0.993651,9.928893,0.983881
8,D1,two_stage,two_stage_habit_fused_equal_session,0.106773,0.102855,0.037011,0.052608,0.050445,0.141140,0.996825,9.938053,0.983564
9,D1,two_stage,two_stage_habit_fused_playtime_session,0.106773,0.102855,0.037441,0.052979,0.050864,0.141110,0.996825,9.936117,0.983511


,method,family,anchor,Hit@10_delta_vs_anchor,NDCG@10_delta_vs_anchor,MRR_delta_vs_anchor
0,popularity_train,anchor,raw,0.07944,0.038090,0.033375
1,popularity_train,anchor,popularity_train,0.00000,0.000000,0.000000
2,fusion_c_raw_plus_behavior,anchor,raw,0.01368,0.003816,0.002480
3,fusion_c_raw_plus_behavior,anchor,popularity_train,-0.06576,-0.034274,-0.030896
4,single_fused_blend_playtime,single_fused,raw,0.01336,0.002024,0.000449
5,single_fused_blend_playtime,single_fused,popularity_train,-0.06608,-0.036065,-0.032926
6,single_fused_blend_equal,single_fused,raw,0.01272,0.001734,0.000191
7,single_fused_blend_equal,single_fused,popularity_train,-0.06672,-0.036356,-0.033184
8,raw,anchor,raw,0.00000,0.000000,0.000000
9,raw,anchor,popularity_train,-0.07944,-0.038090,-0.033375


## 9) Write Summary Artifact + Compact Family Summary

In [13]:
# ===== 9) Write summary artifact + compact family summary =====
OUTPUT_PATH = ARTIFACT_DIR / "eval_two_stage_summary.csv"

summary_tables = {
    "stage1": stage1_table,
    "overall": overall_table,
    "by_slice": by_slice_table,
    "by_support": by_support_table,
    "by_pop_decile": by_pop_decile_table,
    "delta_vs_baselines": delta_vs_baselines_table,
}

for name, table in summary_tables.items():
    print(f"{name:>18}: rows={len(table)}")

combined = []
for name, table in summary_tables.items():
    t = table.copy()
    t.insert(0, "section", name)
    combined.append(t)

summary_artifact = pd.concat(combined, ignore_index=True, sort=False)
summary_artifact.to_csv(OUTPUT_PATH, index=False)
print(f"Wrote summary artifact: {OUTPUT_PATH}")

# Optional compact family-level summary
family_best = (
    overall_table.sort_values(["family", "NDCG@K", "MAP@K", "MRR"], ascending=[True, False, False, False])
    .groupby("family", as_index=False)
    .head(1)
)
display(family_best)

            stage1: rows=5
           overall: rows=11
          by_slice: rows=22
        by_support: rows=11
     by_pop_decile: rows=110
delta_vs_baselines: rows=22
Wrote summary artifact: /home/ryanr/workspace/steam_recommendations/artifacts/recs/eval_two_stage_summary.csv


,family,method,Hit@K,Recall@K,MAP@K,NDCG@K,MRR,ILD@10,CatalogCoverage@10,Novelty@10,PersonalizationGapVsPopularity@10
0,anchor,popularity_train,0.15112,0.146796,0.050594,0.073109,0.073756,0.212335,0.034921,5.317471,0.000000
2,single_fused,single_fused_blend_playtime,0.08504,0.078417,0.023790,0.037044,0.040830,0.132215,1.000000,9.611407,0.972101
5,two_stage,two_stage_behavior_equal_session,0.07232,0.067265,0.023890,0.034341,0.037360,0.139535,1.000000,9.840590,0.980040
